# Train "Hey Andy" (openWakeWord -> TFLite)
Runtime: **GPU (T4 is fine)**. Allow ~1-2 h and ~25 GB disk.

This notebook is a thin wrapper over `scripts/wakeword/run_pipeline.sh`, which pins every package.
Colab's Python is 3.12; the script uses `uv` to fetch its own 3.11, so nothing here touches Colab's packages.
**Not yet run on Colab by the author** - see scripts/wakeword/README.md for what has and has not been verified.

In [ ]:
import subprocess; print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout or "NO GPU - Runtime > Change runtime type")

In [ ]:
BRANCH = "ccr-f3866b26-w6eauq"   # branch holding scripts/wakeword (change to main once merged)
!pip -q install uv
!git clone --depth 1 --branch {BRANCH} https://github.com/BDubDesigns/cough-to-pause.git
%cd cough-to-pause

In [ ]:
# 1. Pinned environments (train + convert) and checksummed mel/embedding models. ~10 min.
!sh scripts/wakeword/run_pipeline.sh setup /content/work

In [ ]:
# 2. Datasets (RIRs, AudioSet noise, FMA music, 2000 h negative features). ~17 GB download.
%cd /content/work
!/content/work/train-env/bin/python -I /content/cough-to-pause/scripts/wakeword/prepare_data.py

In [ ]:
# 3. Generate TTS clips, augment, train, export ONNX, convert to TFLite, verify the Android contract.
#    Exit status is the verdict - if this cell errors, the model is NOT usable.
!sh /content/cough-to-pause/scripts/wakeword/run_pipeline.sh full /content/work

In [ ]:
# 4. Download the three files the app needs.
from google.colab import files
for f in ["/content/work/out/hey_andy.tflite","/content/work/assets/melspectrogram.tflite","/content/work/assets/embedding_model.tflite"]:
    files.download(f)